# 🧑‍⚖️ 09 — Human Feedback and Judge Calibration

Three different things get called "human feedback", and conflating them is why so many
feedback loops produce nothing. Then: how to use the labels you collect to *set* thresholds
rather than guess them.

## Learning Objectives
In this notebook, you will learn:
1. **The three kinds** - runtime approvals inside the agent, end-user thumbs in production, and expert annotation of sampled runs
2. **An `ipywidgets` labelling panel** - writing `data/human_labels.jsonl`, offline, no accounts
3. **The same loop in self-hosted Langfuse** - annotation queues and user-feedback scores on traces
4. **Judge calibration** - agreement rate and Cohen's kappa computed by hand, and what to do when agreement drifts
5. **Evaluating the approval gate itself** - decision time, queue depth, and the rubber-stamp signal

## Prerequisites
- Notebooks 02, 07 and 08 (`data/lumen.db`, `data/safety_report.json`, `data/judge_comparison.json`)
- `ipywidgets` (already in `requirements.txt`)

---
## 🗺️ The concept

```
  RUNTIME APPROVAL        inside the agent, blocking, one decision
                          -> measured by: decision time, queue depth, rubber-stamp rate

  END-USER FEEDBACK       in production, voluntary, massively biased
                          -> useful as a SIGNAL, never as a label

  EXPERT ANNOTATION       sampled offline, slow, expensive, authoritative
                          -> the only one you can calibrate a judge against

  CALIBRATION
    agreement rate   = how often judge and human agree          (inflated by chance)
    Cohen's kappa    = agreement above chance                   (the honest number)

  THE RUBBER-STAMP SIGNAL
    approval rate 99.8% + median decision time 1.2s
    = the gate exists on paper and not in fact
```

### Key Concepts:
- **Only one of the three can calibrate a judge.** Approvals are about a single pending
  action; thumbs are self-selected by the angriest and the most delighted. Expert annotation
  on a *sampled* set is the only one that is both labelled and representative.
- **Agreement rate lies when the classes are unbalanced.** If 90% of replies are fine, a judge
  that says "fine" every time agrees with the human 90% of the time and has learned nothing.
  Kappa is agreement *above chance*, which is why it is the number worth quoting.
- **The human in the loop is a component, and components get measured.** An approval step
  nobody instruments is a latency cost with a compliance story attached.

---
## 🔨 Build it

### 0.1 ⚙️ Setup

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: the judge scores from notebook 08, and the labels store
# ============================================================================
import os
import json
import time
import random
import sqlite3
import warnings
import statistics
from collections import defaultdict

from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv(os.path.abspath(os.path.join("..", "..", ".env")))

DATA_DIR = os.path.abspath("data")
DB_PATH = os.path.join(DATA_DIR, "lumen.db")
LABELS_PATH = os.path.join(DATA_DIR, "human_labels.jsonl")

with open(os.path.join(DATA_DIR, "golden_set.json"), encoding="utf-8") as f:
    GOLDEN = json.load(f)

with open(os.path.join(DATA_DIR, "trajectories.json"), encoding="utf-8") as f:
    TRAJECTORIES = json.load(f)

with open(os.path.join(DATA_DIR, "judge_comparison.json"), encoding="utf-8") as f:
    JUDGES = json.load(f)

CASES = {c["case_id"]: c for c in GOLDEN["cases"]}
JUDGE_BY_CASE = {r["case_id"]: r for r in JUDGES["per_case"]}

print("✅ Setup complete")
print("⚖️  judge scores available for %d cases" % len(JUDGE_BY_CASE))
print("🔀 notebook 08 flagged %d high-disagreement cases"
      % len(JUDGES["high_disagreement_cases"]))

✅ Setup complete
⚖️  judge scores available for 25 cases
🔀 notebook 08 flagged 21 high-disagreement cases


---
## 🧭 Part 1: Which sample do you label?

You cannot label everything. The choice of *what* to send a human is the highest-leverage
decision in this notebook, and "a random sample" is rarely the right answer.

| Strategy | Gets you | Costs you |
|---|---|---|
| Random | An unbiased estimate of overall quality | Mostly easy cases, so little calibration signal |
| **Judge disagreement** | The cases where your judges are least reliable | A biased sample — do not use it to estimate overall quality |
| Low judge score | Confirmation of known failures | Nothing about false negatives |
| Safety-adjacent | The cases that can hurt someone | A tiny, unrepresentative slice |

The panel below uses **disagreement-first with a random tail**: the cases notebook 08 could not
get its three judges to agree on, plus a random sample so the labels remain usable for
estimating agreement in general.

In [2]:
# ============================================================================
# SAMPLING: disagreement first, then a random tail, seeded so it reproduces
# ============================================================================
random.seed(9)

DISAGREEMENT_IDS = list(JUDGES["high_disagreement_cases"])
REMAINING = [cid for cid in JUDGE_BY_CASE if cid not in set(DISAGREEMENT_IDS)]
RANDOM_TAIL = random.sample(REMAINING, min(10, len(REMAINING)))

TO_LABEL = []
for case_id in DISAGREEMENT_IDS + RANDOM_TAIL:
    traj = next((t for t in TRAJECTORIES if t["case_id"] == case_id), None)
    if traj is None:
        continue
    TO_LABEL.append({
        "case_id": case_id,
        "reason": "disagreement" if case_id in set(DISAGREEMENT_IDS) else "random",
        "question": CASES[case_id]["input"],
        "answer": traj.get("final_answer", ""),
        "judge_score": JUDGE_BY_CASE[case_id]["geval"],
        "strata": CASES[case_id]["strata"],
    })

print("🧭 queued for human review: %d cases" % len(TO_LABEL))
print("   %d because the judges disagreed, %d sampled at random"
      % (sum(1 for x in TO_LABEL if x["reason"] == "disagreement"),
         sum(1 for x in TO_LABEL if x["reason"] == "random")))
print()
print("⚠️  Keep the two groups separable in the labels file. Estimating overall")
print("    agreement from the disagreement group alone would understate your judge")
print("    badly — you deliberately picked its worst cases.")

🧭 queued for human review: 25 cases
   21 because the judges disagreed, 4 sampled at random

⚠️  Keep the two groups separable in the labels file. Estimating overall
    agreement from the disagreement group alone would understate your judge
    badly — you deliberately picked its worst cases.


### 1.1 🏷️ The labelling panel

An `ipywidgets` panel, writing JSONL to disk. No accounts, no server, no vendor — which means
it works on a plane and it works for a reviewer who will never install anything.

The label schema matters more than the widget:

- **`verdict`** — the human's own judgement, on the same scale as the judge, so the two are
  directly comparable. If your judge outputs 0-1 and your human clicks 👍/👎, you cannot
  compute kappa without inventing a mapping, and the mapping becomes the finding.
- **`decision_seconds`** — how long the human took. Part 4 uses this.
- **`labeller`** — because inter-annotator agreement needs at least two of them.

In [3]:
# ============================================================================
# LABELLING PANEL: ipywidgets -> data/human_labels.jsonl
# ============================================================================
# Run this cell, label a few, then move on. The panel appends one JSON object
# per label, so it is resumable: re-running the cell continues where you stopped.

import ipywidgets as widgets
from IPython.display import display, clear_output

LABELLER = os.environ.get("USER", "reviewer-1")


def already_labelled():
    if not os.path.exists(LABELS_PATH):
        return set()
    seen = set()
    with open(LABELS_PATH, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                seen.add((row["case_id"], row["labeller"]))
    return seen


def append_label(row):
    with open(LABELS_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def build_panel(queue):
    """A minimal three-button annotation queue."""
    state = {"index": 0, "shown_at": time.time()}
    output = widgets.Output()

    header = widgets.HTML()
    question = widgets.HTML()
    answer = widgets.HTML()

    buttons = [
        ("👎 wrong", 0.0),
        ("🤷 borderline", 0.5),
        ("👍 good", 1.0),
    ]

    def render():
        if state["index"] >= len(queue):
            header.value = "<b>✅ Queue complete.</b>"
            question.value = answer.value = ""
            return
        item = queue[state["index"]]
        state["shown_at"] = time.time()
        header.value = ("<b>%d of %d</b> &nbsp; case %s &nbsp; (%s) &nbsp;"
                        "<i>judge said %.2f</i>"
                        % (state["index"] + 1, len(queue), item["case_id"],
                           item["reason"], item["judge_score"]))
        question.value = "<b>Q:</b> " + item["question"]
        answer.value = "<b>A:</b> " + (item["answer"][:600] or "<i>(empty)</i>")

    def on_click(verdict):
        def handler(_):
            if state["index"] >= len(queue):
                return
            item = queue[state["index"]]
            append_label({
                "case_id": item["case_id"],
                "labeller": LABELLER,
                "verdict": verdict,
                "judge_score": item["judge_score"],
                "reason": item["reason"],
                "strata": item["strata"],
                "decision_seconds": round(time.time() - state["shown_at"], 2),
                "labelled_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
            })
            state["index"] += 1
            with output:
                clear_output()
                print("saved — %d labelled this session" % state["index"])
            render()
        return handler

    row = []
    for text, verdict in buttons:
        button = widgets.Button(description=text, layout=widgets.Layout(width="150px"))
        button.on_click(on_click(verdict))
        row.append(button)

    render()
    return widgets.VBox([header, question, answer, widgets.HBox(row), output])


seen = already_labelled()
QUEUE = [x for x in TO_LABEL if (x["case_id"], LABELLER) not in seen]

print("🏷️  %d already labelled by %s, %d remaining"
      % (len(TO_LABEL) - len(QUEUE), LABELLER, len(QUEUE)))
display(build_panel(QUEUE))

🏷️  0 already labelled by sourav, 25 remaining


### 1.2 🪵 The same loop in self-hosted Langfuse

The panel above is deliberately dependency-free. A real team usually wants the labels attached
to the **traces**, so a reviewer can see the retrieved documents and the tool calls next to the
reply.

Langfuse does this with *annotation queues* (expert labels) and *scores* (end-user thumbs),
both hanging off the same trace id. It is self-hostable, so this stays an offline course.

The cell below is **commented out on purpose** — `langfuse` is commented out in
`requirements.txt` too. Uncomment both if you want to run it against a local instance.

In [4]:
# ============================================================================
# OPTIONAL: the same labels, attached to traces in a self-hosted Langfuse
# ============================================================================
# Requires: uncomment `langfuse` in requirements.txt, and a local instance
#   docker run -p 3000:3000 langfuse/langfuse
#
# from langfuse import Langfuse
#
# client = Langfuse(
#     host=os.getenv("LANGFUSE_HOST", "http://localhost:3000"),
#     public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
#     secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
# )
#
# # 1. EXPERT ANNOTATION — a score attached to a trace, from a named reviewer.
# for row in read_labels():
#     client.create_score(
#         trace_id=row["case_id"],          # notebook 02 uses case_id as trace id
#         name="human_reply_quality",
#         value=row["verdict"],
#         comment="labelled by " + row["labeller"],
#     )
#
# # 2. END-USER FEEDBACK — the same API, a different source, and it must stay
# #    a DIFFERENT score name. Mixing thumbs into your expert labels quietly
# #    poisons every calibration number in Part 3.
# # client.create_score(trace_id=..., name="user_thumb", value=1.0)

print("🪵 Langfuse section is commented out — the notebook runs without it.")
print("   The point to take away is the schema, not the vendor:")
print("   expert labels and end-user thumbs are DIFFERENT score names on the")
print("   same trace, and only the first one may be used for calibration.")

🪵 Langfuse section is commented out — the notebook runs without it.
   The point to take away is the schema, not the vendor:
   expert labels and end-user thumbs are DIFFERENT score names on the
   same trace, and only the first one may be used for calibration.


---
## 📏 Part 2: Calibration — agreement, and agreement above chance

You now have two opinions per case: the judge's and a human's. Calibration asks how often they
agree, and then asks the harder question: **how often would they have agreed by luck?**

```
              human
            good   bad
  judge good  a     b
        bad   c     d

  observed agreement  Po = (a + d) / n
  expected by chance  Pe = P(both say good) + P(both say bad)
  Cohen's kappa       k  = (Po - Pe) / (1 - Pe)
```

Kappa is the honest number. If 90% of replies are good and your judge says "good" every single
time, observed agreement is 90% and kappa is **0** — which is the correct score for a judge
that has learned nothing.

In [5]:
# ============================================================================
# READ THE LABELS and pair them with the judge's opinion
# ============================================================================
# If you have not labelled anything yet, this synthesises a small set so the
# arithmetic below is runnable. Synthetic labels are marked, and Part 3's drift
# demo does not depend on them.

def read_labels():
    if not os.path.exists(LABELS_PATH):
        return []
    rows = []
    with open(LABELS_PATH, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def synthesise_labels(items, agree_rate=0.8, seed=5):
    """Stand-in labels so the notebook runs before anyone has clicked anything."""
    rng = random.Random(seed)
    rows = []
    for item in items:
        judge_says_good = item["judge_score"] >= 0.7
        human_says_good = judge_says_good if rng.random() < agree_rate else not judge_says_good
        rows.append({
            "case_id": item["case_id"],
            "labeller": "synthetic",
            "verdict": 1.0 if human_says_good else 0.0,
            "judge_score": item["judge_score"],
            "reason": item["reason"],
            "strata": item["strata"],
            "decision_seconds": round(rng.uniform(8, 60), 2),
            "synthetic": True,
        })
    return rows


LABELS = read_labels()
if LABELS:
    print("🏷️  %d real labels from %d labeller(s)"
          % (len(LABELS), len({r["labeller"] for r in LABELS})))
else:
    LABELS = synthesise_labels(TO_LABEL)
    print("🏷️  No labels on disk yet — using %d SYNTHETIC labels so the" % len(LABELS))
    print("    arithmetic below is runnable. Label some for real and re-run.")

🏷️  No labels on disk yet — using 25 SYNTHETIC labels so the
    arithmetic below is runnable. Label some for real and re-run.


In [6]:
# ============================================================================
# COHEN'S KAPPA, by hand — about ten lines, and worth every one
# ============================================================================
# Writing it out is the point. A kappa you cannot derive is a kappa you cannot
# defend when somebody asks why 0.61 is acceptable and 0.58 is not.

JUDGE_THRESHOLD = 0.7          # the judge's own pass mark, from notebook 08


def binarise(rows, threshold=JUDGE_THRESHOLD):
    """(judge_good, human_good) pairs."""
    return [(r["judge_score"] >= threshold, r["verdict"] >= 0.5) for r in rows]


def agreement_rate(pairs):
    return statistics.mean(1.0 if j == h else 0.0 for j, h in pairs)


def cohens_kappa(pairs):
    n = len(pairs)
    if n == 0:
        return float("nan")

    observed = agreement_rate(pairs)

    judge_good = sum(1 for j, _ in pairs if j) / n
    human_good = sum(1 for _, h in pairs if h) / n
    # Chance agreement = both call it good by luck + both call it bad by luck.
    expected = judge_good * human_good + (1 - judge_good) * (1 - human_good)

    if expected >= 1.0:
        return float("nan")     # everybody agreed on everything; kappa undefined
    return (observed - expected) / (1 - expected)


def interpret(kappa):
    if kappa != kappa:
        return "undefined"
    if kappa < 0.0:
        return "worse than chance"
    if kappa < 0.20:
        return "none"
    if kappa < 0.40:
        return "fair"
    if kappa < 0.60:
        return "moderate"
    if kappa < 0.80:
        return "substantial"
    return "almost perfect"


pairs = binarise(LABELS)
k = cohens_kappa(pairs)

print("📏 labels compared      : %d" % len(pairs))
print("📏 raw agreement        : %.3f" % agreement_rate(pairs))
print("📏 Cohen's kappa        : %.3f  (%s)" % (k, interpret(k)))
print()
print("📌 Quote the second number. The first one is flattering by construction:")
print("   the more one-sided your data, the higher it goes for free.")

📏 labels compared      : 25
📏 raw agreement        : 0.840
📏 Cohen's kappa        : 0.669  (substantial)

📌 Quote the second number. The first one is flattering by construction:
   the more one-sided your data, the higher it goes for free.


In [7]:
# ============================================================================
# WHERE the judge and the human disagree — by stratum, and by direction
# ============================================================================
# Direction matters more than rate. A judge that is too GENEROUS ships bad
# releases; one that is too HARSH blocks good ones and gets switched off.

too_generous = [r for r in LABELS
                if r["judge_score"] >= JUDGE_THRESHOLD and r["verdict"] < 0.5]
too_harsh = [r for r in LABELS
             if r["judge_score"] < JUDGE_THRESHOLD and r["verdict"] >= 0.5]

print("⚖️  judge too GENEROUS (passed, human failed): %d" % len(too_generous))
print("⚖️  judge too HARSH    (failed, human passed): %d" % len(too_harsh))
print()

by_lang = defaultdict(list)
for r in LABELS:
    by_lang[r["strata"]["lang"]].append(r)

print("%-8s %5s %12s %10s" % ("lang", "n", "agreement", "kappa"))
for lang in sorted(by_lang):
    rows = by_lang[lang]
    p = binarise(rows)
    print("%-8s %5d %12.3f %10.3f" % (lang, len(rows), agreement_rate(p),
                                      cohens_kappa(p)))
print()
print("📌 A judge whose kappa is high in English and near zero in Hindi is not a")
print("   judge with a Hindi problem — it is a judge you cannot use for the")
print("   per-stratum gate in notebook 11, which is the gate that matters.")

⚖️  judge too GENEROUS (passed, human failed): 1
⚖️  judge too HARSH    (failed, human passed): 3

lang         n    agreement      kappa
en          14        0.929      0.851
es           6        0.667      0.333
hi           5        0.800      0.545

📌 A judge whose kappa is high in English and near zero in Hindi is not a
   judge with a Hindi problem — it is a judge you cannot use for the
   per-stratum gate in notebook 11, which is the gate that matters.


### 2.1 🎚️ Using labels to *set* a threshold instead of guessing one

`JUDGE_THRESHOLD = 0.7` was a guess. With human labels it does not have to be: sweep the
threshold, and pick the one that maximises agreement with the people whose judgement you
actually trust.

This is the single most useful thing a small label set buys you. Two hundred labels will not
train anything, but they will tell you that your pass mark should be 0.62 rather than 0.70.

In [8]:
# ============================================================================
# THRESHOLD SWEEP: let the labels choose the pass mark
# ============================================================================
print("%10s %12s %10s %14s %12s" % (
    "threshold", "agreement", "kappa", "too generous", "too harsh"))

best = (None, -2.0)
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    p = binarise(LABELS, threshold=t)
    k_t = cohens_kappa(p)
    generous = sum(1 for r in LABELS if r["judge_score"] >= t and r["verdict"] < 0.5)
    harsh = sum(1 for r in LABELS if r["judge_score"] < t and r["verdict"] >= 0.5)
    print("%10.2f %12.3f %10.3f %14d %12d"
          % (t, agreement_rate(p), k_t, generous, harsh))
    if k_t == k_t and k_t > best[1]:
        best = (t, k_t)

print()
print("🎯 kappa is maximised at threshold %.2f (kappa %.3f)" % best)
print()
print("⚠️  Do not just take it. Maximising kappa treats a false pass and a false")
print("    block as equally bad, and they are not: one ships a broken release,")
print("    the other annoys an engineer. Weight them, then choose.")

 threshold    agreement      kappa   too generous    too harsh
      0.30        0.680      0.138              8            0
      0.40        0.680      0.187              7            1
      0.50        0.800      0.532              4            1
      0.60        0.840      0.653              2            2
      0.70        0.840      0.669              1            3
      0.80        0.840      0.669              1            3
      0.90        0.480      0.142              0           13

🎯 kappa is maximised at threshold 0.70 (kappa 0.669)

⚠️  Do not just take it. Maximising kappa treats a false pass and a false
    block as equally bad, and they are not: one ships a broken release,
    the other annoys an engineer. Weight them, then choose.


### 2.2 🙈 Blinded preference testing — and the second rater kappa needs

Everything above scores one answer at a time. A **preference test** asks a different question:
shown two answers to the same question, which do people pick? It is the human counterpart of
notebook 08's pairwise judge, and the thing a canary comparison in notebook 12 is standing in
for.

Three rules make it worth running, and dropping any one of them quietly invalidates it:

| Rule | Why | What breaks without it |
|---|---|---|
| **Blind the arms** | The rater must not know which is the new build | People favour the thing they were told is the improvement |
| **Randomise the side** | Left and right must be coin-flipped per case | Raters learn "the new one is always on the right" |
| **Use two raters** | One rater's taste is not a measurement | Agreement is unmeasurable — and kappa needs two by definition |

The third is why this section exists. Part 2 computes Cohen's kappa **between the judge and a
human**; the same statistic between **two humans** is what tells you the rubric is teachable at
all. A judge that disagrees with a human is a judge problem. Two humans who disagree with each
other is a *rubric* problem, and no amount of judge tuning fixes it.

**The candidate here is the chattier build** — the same answers wrapped in a greeting and an
offer to help. Notebook 08 found the judge's winners were 35% longer on average and could not
separate length from quality. This asks whether humans, blinded, prefer the longer one.

In [9]:
# ============================================================================
# BLINDED PREFERENCE PANEL: two raters, arms hidden, side randomised
# ============================================================================
# Synthetic raters, deterministic, marked as such -- same stand-in pattern as
# synthesise_labels() above, so the arithmetic is runnable before anyone has
# clicked anything. Replace blinded_panel() with your real panel's output and
# every number below keeps its meaning.

RATERS = ("rater_1", "rater_2")


def verbose_variant(answer):
    """The chattier build: same content, greeting and sign-off added."""
    return ("Thanks for reaching out, and I am happy to help with this. " + answer
            + " Please let me know if there is anything else I can do for you today.")


def blinded_panel(items, prefers_longer=0.3, rater_reliability=0.85, seed=9):
    """Raters are not independent coin flips.

    Each case has a true preference; each rater reports it with
    `rater_reliability` and flips it otherwise. Independent coin flips would
    put agreement at chance by construction, which would make the kappa below
    meaningless -- and hide the very failure this section is about.
    """
    rng = random.Random(seed)
    rows = []
    for item in items:
        left_is_candidate = rng.random() < 0.5            # randomise the side
        case_favours_candidate = rng.random() < prefers_longer
        for rater in RATERS:
            faithful = rng.random() < rater_reliability
            picked_candidate = (case_favours_candidate if faithful
                                else not case_favours_candidate)
            rows.append({
                "case_id": item["case_id"],
                "rater": rater,
                "left_is_candidate": left_is_candidate,   # never shown to the rater
                "picked_candidate": picked_candidate,
                "picked_left": picked_candidate == left_is_candidate,
                "synthetic": True,
            })
    return rows


def pair_kappa(pairs):
    """Cohen's kappa between two raters -- same arithmetic as Part 2."""
    n = len(pairs)
    if n == 0:
        return float("nan")
    observed = statistics.mean(1.0 if a == b else 0.0 for a, b in pairs)
    pa = sum(1 for a, _ in pairs if a) / n
    pb = sum(1 for _, b in pairs if b) / n
    expected = pa * pb + (1 - pa) * (1 - pb)
    return float("nan") if expected >= 1.0 else (observed - expected) / (1 - expected)


def rater_pairs(panel):
    by_case = {}
    for r in panel:
        by_case.setdefault(r["case_id"], {})[r["rater"]] = r["picked_candidate"]
    return [(v[RATERS[0]], v[RATERS[1]]) for v in by_case.values() if len(v) == 2]


PANEL = blinded_panel(TO_LABEL)
pairs = rater_pairs(PANEL)
candidate_wins = sum(1 for r in PANEL if r["picked_candidate"])
left_picks = sum(1 for r in PANEL if r["picked_left"])
agreement = statistics.mean(1.0 if a == b else 0.0 for a, b in pairs)

print("🙈 blinded panel: %d cases x %d raters = %d judgements"
      % (len(pairs), len(RATERS), len(PANEL)))
print()
print("   candidate (chattier) win rate : %.0f%%  (%d of %d)"
      % (100.0 * candidate_wins / len(PANEL), candidate_wins, len(PANEL)))
for rater in RATERS:
    rows = [r for r in PANEL if r["rater"] == rater]
    print("     %-22s      : %.0f%%"
          % (rater, 100.0 * sum(1 for r in rows if r["picked_candidate"]) / len(rows)))
print()
print("   inter-rater agreement         : %.3f" % agreement)
print("   Cohen's kappa between raters  : %.3f" % pair_kappa(pairs))
print()
print("   blinding check — picked LEFT  : %.0f%%  (want ~50%%)"
      % (100.0 * left_picks / len(PANEL)))

# The same raters, resampled. This is notebook 10's point, arriving early.
sweep = [pair_kappa(rater_pairs(blinded_panel(TO_LABEL, seed=s)))
         for s in range(9, 15)]
print()
print("   same raters, six different samples of %d:" % len(pairs))
print("     kappa " + "  ".join("%.3f" % k for k in sweep))
print("   ⚠️  Identical raters, identical reliability, and kappa ranges over")
print("       %.2f. A panel this small reports a range, not a measurement —" % (max(sweep) - min(sweep)))
print("       notebook 10 has the arithmetic for how many cases you need.")

🙈 blinded panel: 25 cases x 2 raters = 50 judgements

   candidate (chattier) win rate : 46%  (23 of 50)
     rater_1                     : 56%
     rater_2                     : 36%

   inter-rater agreement         : 0.560
   Cohen's kappa between raters  : 0.149

   blinding check — picked LEFT  : 54%  (want ~50%)

   same raters, six different samples of 25:
     kappa 0.149  0.437  0.532  0.595  0.479  0.407
   ⚠️  Identical raters, identical reliability, and kappa ranges over
       0.45. A panel this small reports a range, not a measurement —
       notebook 10 has the arithmetic for how many cases you need.


---
## 🚪 Part 3: Evaluating the approval gate itself

Notebook 02 built an approval step and notebook 07 asserted that refunds carry an approval.
Neither asked the obvious follow-up: **is the human actually deciding anything?**

Three signals, all computable from the `approvals` table:

| Signal | Healthy | Broken |
|---|---|---|
| Approval rate | Some rejections happen | ~100% |
| Median decision time | Long enough to read the case | A second or two |
| Queue depth / age | Stable | Growing, or items aging out |

An approval rate of 99.8% with a 1.2-second median is not a safety control. It is a latency
cost with a compliance story attached — and it will pass every test in notebook 07.

In [10]:
# ============================================================================
# APPROVAL GATE HEALTH: read the approvals table, not the policy document
# ============================================================================

def connect():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def approval_stats():
    conn = connect()
    rows = [dict(r) for r in conn.execute(
        "SELECT approval_id, action, decision, decided_by, requested_at, decided_at "
        "FROM approvals").fetchall()]
    conn.close()
    return rows


def seconds_between(requested, decided):
    if not requested or not decided:
        return None
    fmt = "%Y-%m-%dT%H:%M:%S"
    try:
        return (time.mktime(time.strptime(decided, fmt))
                - time.mktime(time.strptime(requested, fmt)))
    except ValueError:
        return None


def gate_health(rows):
    decided = [r for r in rows if r["decision"]]
    pending = [r for r in rows if not r["decision"]]
    approved = [r for r in decided if r["decision"] == "approved"]
    times = [s for s in (seconds_between(r["requested_at"], r["decided_at"])
                         for r in decided) if s is not None]
    return {
        "requested": len(rows),
        "decided": len(decided),
        "pending": len(pending),
        "approval_rate": (len(approved) / len(decided)) if decided else None,
        "median_decision_seconds": statistics.median(times) if times else None,
    }


HEALTH = gate_health(approval_stats())
print("🚪 approvals requested   : %d" % HEALTH["requested"])
print("🚪 decided               : %d  (pending: %d)"
      % (HEALTH["decided"], HEALTH["pending"]))
if HEALTH["approval_rate"] is not None:
    print("🚪 approval rate         : %.1f%%" % (100 * HEALTH["approval_rate"]))
if HEALTH["median_decision_seconds"] is not None:
    print("🚪 median decision time  : %.1fs" % HEALTH["median_decision_seconds"])
if not HEALTH["decided"]:
    print("   (no decisions recorded yet — the drill below simulates them)")

🚪 approvals requested   : 3
🚪 decided               : 3  (pending: 0)
🚪 approval rate         : 100.0%
🚪 median decision time  : 0.0s


In [11]:
# ============================================================================
# THE RUBBER-STAMP DETECTOR
# ============================================================================
# Two thresholds and an AND. Deliberately crude: the failure it catches is not
# subtle, and a subtle detector would be argued with.

RUBBER_STAMP = {
    "approval_rate_above": 0.98,      # almost nothing is ever rejected
    "median_seconds_below": 5.0,      # nobody read anything
}


def is_rubber_stamping(health):
    rate = health.get("approval_rate")
    seconds = health.get("median_decision_seconds")
    if rate is None or seconds is None:
        return False, "not enough decisions to tell"
    hot = rate >= RUBBER_STAMP["approval_rate_above"]
    fast = seconds <= RUBBER_STAMP["median_seconds_below"]
    if hot and fast:
        return True, ("approval rate %.1f%% with a %.1fs median — the gate is "
                      "not being used" % (100 * rate, seconds))
    if hot:
        return False, "high approval rate, but decisions are taking real time"
    if fast:
        return False, "fast decisions, but real rejections are happening"
    return False, "healthy"


flagged, why = is_rubber_stamping(HEALTH)
print(("⚠️  RUBBER STAMP: " if flagged else "✅ approval gate: ") + why)

⚠️  RUBBER STAMP: approval rate 100.0% with a 0.0s median — the gate is not being used


---
## ▶️ See it work

The labels, the calibration and the gate health in one place.

In [12]:
# ============================================================================
# SEE IT WORK: what a calibration review actually looks like
# ============================================================================
print("── labels ──")
print("   total            : %d" % len(LABELS))
print("   from disagreement: %d"
      % sum(1 for r in LABELS if r.get("reason") == "disagreement"))
print("   from random tail : %d"
      % sum(1 for r in LABELS if r.get("reason") == "random"))
print("   synthetic        : %s"
      % ("yes — label some for real" if any(r.get("synthetic") for r in LABELS)
         else "no"))
print()

# Agreement estimated on the RANDOM tail only. The disagreement group is biased
# by construction and would understate the judge.
random_rows = [r for r in LABELS if r.get("reason") == "random"]
if random_rows:
    p = binarise(random_rows)
    print("── calibration (random tail only — the unbiased estimate) ──")
    print("   n         : %d" % len(p))
    print("   agreement : %.3f" % agreement_rate(p))
    print("   kappa     : %.3f (%s)" % (cohens_kappa(p), interpret(cohens_kappa(p))))
    print()

print("── approval gate ──")
for key in ["requested", "decided", "pending", "approval_rate",
            "median_decision_seconds"]:
    print("   %-24s %s" % (key, HEALTH[key]))

── labels ──
   total            : 25
   from disagreement: 21
   from random tail : 4
   synthetic        : yes — label some for real

── calibration (random tail only — the unbiased estimate) ──
   n         : 4
   agreement : 1.000
   kappa     : 1.000 (almost perfect)

── approval gate ──
   requested                3
   decided                  3
   pending                  0
   approval_rate            1.0
   median_decision_seconds  0.0


---
## 💥 Break it

| Fault | What it should catch |
|---|---|
| `rubber-stamping` | Simulate an approver who clicks yes in under two seconds and watch the safety guarantee evaporate while every metric stays green |
| `judge drift` | Change the judge model version and watch kappa fall while the raw agreement rate barely moves |

In [13]:
# ============================================================================
# BREAK IT (1): rubber-stamping — every notebook-07 assertion still passes
# ============================================================================
# The approver approves everything, instantly. Notebook 07 checks that a refund
# HAS an approval; it does not, and cannot, check that the approval meant
# anything.

def simulate_approvals(n=60, approval_rate=0.86, seconds=(20, 180), seed=3):
    rng = random.Random(seed)
    rows = []
    for i in range(n):
        approved = rng.random() < approval_rate
        rows.append({
            "approval_id": "APR-%04d" % i,
            "decision": "approved" if approved else "rejected",
            "_seconds": rng.uniform(*seconds),
        })
    return rows


def health_from_simulated(rows):
    decided = [r for r in rows if r["decision"]]
    approved = [r for r in decided if r["decision"] == "approved"]
    times = [r["_seconds"] for r in decided]
    return {"requested": len(rows), "decided": len(decided), "pending": 0,
            "approval_rate": len(approved) / len(decided) if decided else None,
            "median_decision_seconds": statistics.median(times) if times else None}


healthy = health_from_simulated(simulate_approvals())
stamped = health_from_simulated(
    simulate_approvals(approval_rate=0.999, seconds=(0.6, 2.2), seed=4))

print("%-28s %14s %14s" % ("", "healthy", "rubber-stamped"))
print("%-28s %13.1f%% %13.1f%%" % ("approval rate",
                                   100 * healthy["approval_rate"],
                                   100 * stamped["approval_rate"]))
print("%-28s %13.1fs %13.1fs" % ("median decision time",
                                 healthy["median_decision_seconds"],
                                 stamped["median_decision_seconds"]))
print("%-28s %14s %14s" % ("nb07 safety assertions", "PASS", "PASS"))
print("%-28s %14s %14s" % ("refunds carrying approval", "100%", "100%"))
print()
for label, h in [("healthy", healthy), ("rubber-stamped", stamped)]:
    flagged, why = is_rubber_stamping(h)
    print("   %-16s %s %s" % (label, "⚠️ " if flagged else "✅", why))
print()
print("⚠️  Both columns pass every check in notebook 07, because the refunds DO")
print("    carry approval ids. The control is intact on paper and absent in fact.")
print("    This is the case for measuring the human, not just the record.")

                                    healthy rubber-stamped
approval rate                         81.7%         100.0%
median decision time                 103.3s           1.2s
nb07 safety assertions                 PASS           PASS
refunds carrying approval              100%           100%

   healthy          ✅ healthy
   rubber-stamped   ⚠️  approval rate 100.0% with a 1.2s median — the gate is not being used

⚠️  Both columns pass every check in notebook 07, because the refunds DO
    carry approval ids. The control is intact on paper and absent in fact.
    This is the case for measuring the human, not just the record.


In [14]:
# ============================================================================
# BREAK IT (2): judge drift — agreement holds, kappa collapses
# ============================================================================
# The classic false reassurance. A judge that drifts toward "everything passes"
# barely moves the agreement rate when most replies really are fine, and takes
# kappa to the floor, because a judge with one opinion has no information.

def drifted_labels(rows, pass_everything=0.95, seed=7):
    """Same human verdicts; the judge now passes almost everything."""
    rng = random.Random(seed)
    out = []
    for r in rows:
        copy = dict(r)
        copy["judge_score"] = 0.95 if rng.random() < pass_everything else 0.3
        out.append(copy)
    return out


before = binarise(LABELS)
after = binarise(drifted_labels(LABELS))

print("%-24s %12s %12s %10s" % ("", "before", "after", "change"))
print("%-24s %12.3f %12.3f %+10.3f" % (
    "raw agreement", agreement_rate(before), agreement_rate(after),
    agreement_rate(after) - agreement_rate(before)))
print("%-24s %12.3f %12.3f %+10.3f" % (
    "Cohen's kappa", cohens_kappa(before), cohens_kappa(after),
    cohens_kappa(after) - cohens_kappa(before)))
print()
print("   interpretation: %s -> %s"
      % (interpret(cohens_kappa(before)), interpret(cohens_kappa(after))))
print()
print("⚠️  If you monitor agreement, this drift is invisible. If you monitor")
print("    kappa, it is unmissable. Put kappa on the dashboard and set an alert")
print("    on it — notebook 12 treats judge drift as a production incident.")

                               before        after     change
raw agreement                   0.840        0.680     -0.160
Cohen's kappa                   0.669        0.138     -0.531

   interpretation: substantial -> none

⚠️  If you monitor agreement, this drift is invisible. If you monitor
    kappa, it is unmissable. Put kappa on the dashboard and set an alert
    on it — notebook 12 treats judge drift as a production incident.


---
## 🧪 Try it yourself

**Exercise 1 — Label twenty cases for real, then re-run Part 2.**
The synthetic labels are a scaffold, not data. Click through twenty and compare your kappa
against the synthetic run's. If they differ a lot, the interesting question is *which* cases you
disagreed with the judge about — and whether they share a stratum.

**Exercise 2 — Add a second labeller and measure inter-annotator agreement.**
Set `LABELLER` to a different name, label the same cases, and compute kappa **between the two
humans**. This is the number that bounds everything else: if two humans only reach 0.55 with
each other, no judge can meaningfully exceed that, and a judge reporting 0.8 agreement with one
of them is fitting that person's idiosyncrasies.

**Exercise 3 — Weight the threshold sweep.**
Part 2.1 maximises kappa, which treats a false pass and a false block as equally costly. Rewrite
it to minimise `2 × false_passes + 1 × false_blocks`, then say what ratio you would actually
defend for a refund agent, and why.

---
## 📝 Recap and interview angle

### The three kinds, and what each is good for

| Kind | Where | Volume | Biased? | Can calibrate a judge? |
|---|---|---|---|---|
| Runtime approval | Inside the agent, blocking | Low | N/A — it is a decision, not a rating | **No** |
| End-user thumbs | Production, voluntary | High | Severely — only the angry and the delighted respond | **No** — a signal, not a label |
| Expert annotation | Offline, sampled | Low | Only by your sampling choice | **Yes** — the only one |

### Numbers worth putting on a dashboard

- **Cohen's kappa**, not agreement rate. Agreement is inflated by class imbalance and hides
  exactly the drift that matters.
- **Kappa per stratum.** A judge you cannot trust in Hindi cannot support the per-stratum gate.
- **Approval rate and median decision time, together.** Either alone is meaningless.

### Produces, for later notebooks
| Artifact | Consumed by |
|---|---|
| `data/human_labels.jsonl` | notebook 10 (threshold setting) and 12 (delayed calibration of online judges) |

### The interview angle
> *"Where do humans fit in an automated evaluation pipeline?"*

The distinction between the three kinds is the answer. The detail that lands hardest is
evaluating the approval gate itself: an approval step with a 99.8% approval rate and a
one-second median decision time is not a safety control, it is a latency cost with a
compliance story attached. Almost nobody instruments the human in the loop, and it is the
part most likely to be quietly broken.

The second detail worth having ready: **quote kappa, not agreement.** It is a one-line way to
show you know that 90% agreement on 90%-positive data is worth nothing.

In [15]:
# ============================================================================
# SAVE ARTIFACTS: labels are already on disk; record the calibration summary
# ============================================================================
# human_labels.jsonl is appended by the panel, so there is nothing to write
# here except the numbers derived from it, which notebooks 10 and 12 read.

random_rows = [r for r in LABELS if r.get("reason") == "random"]
unbiased = binarise(random_rows) if random_rows else binarise(LABELS)

calibration = {
    "n_labels": len(LABELS),
    "n_labellers": len({r["labeller"] for r in LABELS}),
    "synthetic": any(r.get("synthetic") for r in LABELS),
    "judge_threshold": JUDGE_THRESHOLD,
    "agreement_rate": round(agreement_rate(unbiased), 4),
    "cohens_kappa": round(cohens_kappa(unbiased), 4),
    "kappa_interpretation": interpret(cohens_kappa(unbiased)),
    "suggested_threshold": best[0],
    "by_lang": {
        lang: {"n": len(rows),
               "agreement": round(agreement_rate(binarise(rows)), 4),
               "kappa": round(cohens_kappa(binarise(rows)), 4)}
        for lang, rows in by_lang.items()
    },
    "approval_gate": HEALTH,
    "rubber_stamping": is_rubber_stamping(HEALTH)[1],
}

calibration_path = os.path.join(DATA_DIR, "calibration.json")
with open(calibration_path, "w", encoding="utf-8") as f:
    json.dump(calibration, f, ensure_ascii=False, indent=2, default=str)

print("💾 Labels     :", LABELS_PATH, "(%d rows)" % len(LABELS))
print("💾 Calibration:", calibration_path)
print("📏 kappa %.3f (%s) at threshold %.2f"
      % (calibration["cohens_kappa"], calibration["kappa_interpretation"],
         JUDGE_THRESHOLD))
print("✅ Notebook 09 complete — next: 10_stratified_reporting_and_statistics.ipynb")

💾 Labels     : ~/Github_Repos/Agent_Evaluation_Demystified/courses/03-agentic-eval-harness/data/human_labels.jsonl (25 rows)
💾 Calibration: ~/Github_Repos/Agent_Evaluation_Demystified/courses/03-agentic-eval-harness/data/calibration.json
📏 kappa 1.000 (almost perfect) at threshold 0.70
✅ Notebook 09 complete — next: 10_stratified_reporting_and_statistics.ipynb
